# Selection Analysis

---

This notebook has two parts:

**Part 1 — Non-competitive vs. Competitive Schools:** Do students nominated at institutions with 0% or 100% acceptance rates differ systematically from students at competitive institutions? An LPM predicts whether a student attends a non-competitive school using individual characteristics, establishing whether the sample restriction is a capacity decision or a selection-on-observables decision.

**Part 2 — Selection within Competitive Schools:** What individual characteristics predict acceptance at the 23 institutions where selection is genuinely occurring? Primary model is an LPM with institution fixed effects and clustered standard errors. Logistic regression is reported in the appendix as a robustness check.

In [8]:
import sys
sys.path.insert(0, '..')

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

from src.model_acceptance import (
    load_data, engineer_student_features, engineer_military_features,
    engineer_ipeds_features, merge_ipeds, compute_icc, OUTCOME,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.3f}'.format)

COVARS = ['GPA', 'female', 'pell', 'veteran', 'student_year', 'grad_student',
          'mil_affil']

LABELS = {
    'GPA': 'GPA (mean-centered)',
    'female': 'Female',
    'pell':         'Pell Grant',
    'veteran':      'Veteran',
    'student_year': 'Student Year',
    'grad_student': 'Graduate Student',
    'mil_affil':    'Military Affiliation',
}

def sig_stars(p):
    if pd.isna(p): return ''
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))

In [9]:
# Load, engineer, and define samples 
vsa_raw, ipeds_raw = load_data()
vsa   = engineer_student_features(vsa_raw)
vsa   = engineer_military_features(vsa)
ipeds = engineer_ipeds_features(ipeds_raw)
df    = merge_ipeds(vsa, ipeds)

df_eligible = df[
    (df['GPA'] >= 3.2) &
    (df['us_citizen'] == 1) &
    (~df['other_institution_flag'])
].copy()

# Count selected and eligible only among eligible applicants so rates ≤ 1
selected_by_inst = df_eligible.groupby('Institution')[OUTCOME].sum().rename('n_selected')
eligible_by_inst = df_eligible.groupby('Institution').size().rename('n_eligible')

inst_stats = (
    pd.concat([selected_by_inst, eligible_by_inst], axis=1)
    .assign(acceptance_rate=lambda d: d['n_selected'] / d['n_eligible'])
    .sort_values('acceptance_rate')
    .reset_index()
)

competitive_insts = inst_stats[(inst_stats['acceptance_rate'] > 0) &
                                    (inst_stats['acceptance_rate'] < 1)]['Institution']
non_competitive_insts = inst_stats[(inst_stats['acceptance_rate'] == 0) |
                                    (inst_stats['acceptance_rate'] == 1)]['Institution']

# Tag each eligible student
df_eligible = df_eligible.copy()
df_eligible['non_competitive'] = df_eligible['Institution'].isin(non_competitive_insts).astype(int)

# Center GPA on the full eligible sample mean — consistent across both parts
gpa_mean_eligible = df_eligible['GPA'].mean()
df_eligible['GPA'] = df_eligible['GPA'] - gpa_mean_eligible
print(f'GPA centered on eligible sample mean: {gpa_mean_eligible:.3f}')

# Both samples share the same centering
df_full        = df_eligible.copy()
df_competitive = df_eligible[df_eligible['Institution'].isin(competitive_insts)].copy()

n_zero = (inst_stats['acceptance_rate'] == 0).sum()
n_full_insts = (inst_stats['acceptance_rate'] == 1).sum()
n_comp = len(competitive_insts)

print(f'Full eligible sample:      {df_eligible.shape[0]:,} applicants, {df_eligible["Institution"].nunique()} institutions')
print(f'Non-competitive (0%/100%): {df_eligible["non_competitive"].sum():,} applicants, {n_zero + n_full_insts} institutions ({n_zero} at 0%, {n_full_insts} at 100%)')
print(f'Competitive (0%–100%):     {df_competitive.shape[0]:,} applicants, {n_comp} institutions')

GPA centered on eligible sample mean: 3.680
Full eligible sample:      841 applicants, 41 institutions
Non-competitive (0%/100%): 166 applicants, 13 institutions (1 at 0%, 12 at 100%)
Competitive (0%–100%):     675 applicants, 28 institutions


---
## Part 1: Non-competitive vs. Competitive Schools

### 1.1 Descriptive Comparison

Mean of each predictor by institution type, with a t-test for difference in means.

In [10]:
rows = []
sub = df_full[COVARS + ['non_competitive']].dropna()

comp_grp    = sub[sub['non_competitive'] == 0]
noncomp_grp = sub[sub['non_competitive'] == 1]

for col in COVARS:
    c = comp_grp[col].dropna()
    n = noncomp_grp[col].dropna()
    t, p = stats.ttest_ind(c, n, equal_var=False)
    rows.append({
        'Predictor':         LABELS[col],
        'Competitive (mean)':     round(c.mean(), 3),
        'Non-competitive (mean)': round(n.mean(), 3),
        'Difference':        round(n.mean() - c.mean(), 3),
        'p-value':           round(p, 3),
        'sig':               sig_stars(p),
    })

desc_tbl = pd.DataFrame(rows).set_index('Predictor')
print(f'Competitive: n={len(comp_grp):,}   Non-competitive: n={len(noncomp_grp):,}')
print('─' * 70)
display(desc_tbl)

Competitive: n=604   Non-competitive: n=146
──────────────────────────────────────────────────────────────────────


,Competitive (mean),Non-competitive (mean),Difference,p-value,sig
Predictor,,,,,
GPA (mean-centered),-0.015,0.026,0.042,0.058,
Female,0.286,0.240,-0.047,0.244,
Pell Grant,0.396,0.384,-0.012,0.788,
Veteran,0.060,0.041,-0.019,0.333,
Student Year,2.826,2.685,-0.141,0.176,
Graduate Student,0.101,0.137,0.036,0.248,
Military Affiliation,0.091,0.089,-0.002,0.939,


### 1.2 LPM - Predicting Non-competitive School Attendance

Outcome: `non_competitive` (1 = student is at a 0% or 100% acceptance institution). Standard errors clustered by institution. No institution fixed effects — institution type is what we are explaining.

In [11]:
fixed   = ' + '.join(COVARS)
formula = f'non_competitive ~ {fixed}'

lpm_nc = smf.ols(formula, data=df_full.dropna(subset=COVARS + ['non_competitive'])).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_full.dropna(subset=COVARS + ['non_competitive'])['Institution']},
    disp=False,
)

params = lpm_nc.params[COVARS]
ci     = lpm_nc.conf_int().loc[COVARS]
pvals  = lpm_nc.pvalues[COVARS]

nc_tbl = pd.DataFrame({
    'Coef':     params,
    'CI_lower': ci.iloc[:, 0],
    'CI_upper': ci.iloc[:, 1],
    'p_value':  pvals,
})
nc_tbl['CI_95'] = (nc_tbl['CI_lower'].map('{:.3f}'.format) +
                   ' – ' + nc_tbl['CI_upper'].map('{:.3f}'.format))
nc_tbl['sig']   = nc_tbl['p_value'].apply(sig_stars)
nc_tbl.index    = [LABELS[c] for c in COVARS]

print(f'LPM — outcome: non-competitive school (n={lpm_nc.nobs:.0f}, R²={lpm_nc.rsquared:.3f})')
print('Clustered SEs by institution. Coef = p.p. change in P(non-competitive).')
print('─' * 60)
display(nc_tbl[['Coef', 'CI_95', 'p_value', 'sig']].round(3))

# Verify coefficients are within [-1, 1]
out_of_range = nc_tbl['Coef'].abs() > 1
if out_of_range.any():
    print(f'\n⚠  Coefficients outside [-1, 1]: {out_of_range[out_of_range].index.tolist()}')

LPM — outcome: non-competitive school (n=750, R²=0.011)
Clustered SEs by institution. Coef = p.p. change in P(non-competitive).
────────────────────────────────────────────────────────────


,Coef,CI_95,p_value,sig
GPA (mean-centered),0.099,-0.117 – 0.315,0.367,
Female,-0.042,-0.105 – 0.022,0.197,
Pell Grant,0.006,-0.073 – 0.086,0.878,
Veteran,-0.054,-0.188 – 0.080,0.429,
Student Year,-0.014,-0.048 – 0.019,0.403,
Graduate Student,0.047,-0.069 – 0.162,0.431,
Military Affiliation,-0.005,-0.141 – 0.131,0.940,


---
## Part 2: Selection within Competitive Schools

### 2.1 Primary Model — LPM with Institution Fixed Effects

Outcome: `accepted`. Institution fixed effects absorb between-school differences. Standard errors clustered by institution. Coefficients are bounded between −1 and 1 and interpreted as percentage-point changes in acceptance probability.

In [12]:
def fit_fe_model(df, outcome, covars, model='ols'):
    cols = [outcome, 'Institution'] + covars
    sub = df[cols].dropna().copy()
    fixed  = ' + '.join(covars)
    formula = f'{outcome} ~ {fixed} + C(Institution)'
    fit_fn  = smf.logit if model == 'logit' else smf.ols
    result  = fit_fn(formula, data=sub).fit(
        cov_type='cluster',
        cov_kwds={'groups': sub['Institution']},
        disp=False,
    )
    return result, sub


def lpm_table(result, covars):
    params = result.params[covars]
    ci     = result.conf_int().loc[covars]
    pvals  = result.pvalues[covars]
    tbl = pd.DataFrame({'Coef': params, 'CI_lower': ci.iloc[:,0],
                         'CI_upper': ci.iloc[:,1], 'p_value': pvals})
    tbl['CI_95'] = (tbl['CI_lower'].map('{:.3f}'.format) +
                    ' – ' + tbl['CI_upper'].map('{:.3f}'.format))
    tbl['sig']   = tbl['p_value'].apply(sig_stars)
    tbl.index    = [LABELS[c] for c in covars]
    return tbl[['Coef', 'CI_95', 'p_value', 'sig']].round(3)


def logit_table(result, covars):
    params = result.params[covars]
    ci     = result.conf_int().loc[covars]
    pvals  = result.pvalues[covars]
    tbl = pd.DataFrame({'OR': np.exp(params), 'CI_lower': np.exp(ci.iloc[:,0]),
                         'CI_upper': np.exp(ci.iloc[:,1]), 'p_value': pvals})
    tbl['CI_95'] = (tbl['CI_lower'].map('{:.3f}'.format) +
                    ' – ' + tbl['CI_upper'].map('{:.3f}'.format))
    tbl['sig']   = tbl['p_value'].apply(sig_stars)
    tbl.index    = [LABELS[c] for c in covars]
    return tbl[['OR', 'CI_95', 'p_value', 'sig']].round(3)


lpm_result, lpm_sub = fit_fe_model(df_competitive, OUTCOME, COVARS, model='ols')
lpm_tbl = lpm_table(lpm_result, COVARS)

print(f'LPM — institution fixed effects')
print(f'n = {lpm_result.nobs:.0f}, R² = {lpm_result.rsquared:.3f}, AIC = {lpm_result.aic:.2f}')
print('Clustered SEs by institution. GPA mean-centered.')
print('─' * 60)
display(lpm_tbl)

# Verify LPM coefficients within [-1, 1]
oor = lpm_result.params[COVARS].abs() > 1
if oor.any():
    print(f'\n⚠  Coefficients outside [-1, 1]: {oor[oor].index.tolist()}')
else:
    print('\nAll coefficients within [-1, 1]. ✓')

# Out-of-range fitted values
preds = lpm_result.fittedvalues
n_oor = ((preds < 0) | (preds > 1)).sum()
if n_oor > 0:
    print(f'⚠  {n_oor} fitted values outside [0,1] ({n_oor/len(preds):.1%} of sample).')

LPM — institution fixed effects
n = 604, R² = 0.372, AIC = 609.50
Clustered SEs by institution. GPA mean-centered.
────────────────────────────────────────────────────────────


,Coef,CI_95,p_value,sig
GPA (mean-centered),0.149,-0.016 – 0.314,0.077,
Female,0.025,-0.040 – 0.091,0.451,
Pell Grant,0.031,-0.018 – 0.079,0.215,
Veteran,0.092,-0.029 – 0.214,0.137,
Student Year,0.020,-0.011 – 0.052,0.210,
Graduate Student,-0.077,-0.194 – 0.040,0.195,
Military Affiliation,-0.220,-0.429 – -0.012,0.039,*



All coefficients within [-1, 1]. ✓
⚠  60 fitted values outside [0,1] (9.9% of sample).


### 2.2 Appendix — Logistic Regression with Institution Fixed Effects

Same specification as the LPM. Coefficients reported as odds ratios. Included as a robustness check on functional form.

In [13]:
logit_result, _ = fit_fe_model(df_competitive, OUTCOME, COVARS, model='logit')
logit_tbl = logit_table(logit_result, COVARS)

print(f'Logistic regression — institution fixed effects')
print(f'n = {logit_result.nobs:.0f}, pseudo-R² = {logit_result.prsquared:.3f}, AIC = {logit_result.aic:.2f}')
print('Clustered SEs by institution. GPA mean-centered.')
print('─' * 60)
display(logit_tbl)

# Side-by-side with LPM
comparison = pd.DataFrame({
    'Coef (LPM)':  lpm_tbl['Coef'],
    'sig (LPM)':   lpm_tbl['sig'],
    'OR (Logit)':  logit_tbl['OR'],
    'sig (Logit)': logit_tbl['sig'],
})
print('\nLPM vs. Logit comparison')
print('─' * 55)
display(comparison.round(3))

changed = comparison[comparison['sig (LPM)'] != comparison['sig (Logit)']]
if len(changed):
    print('\nPredictors where significance differs:')
    display(changed.round(3))
else:
    print('\nNo predictors change significance across specifications.')

Logistic regression — institution fixed effects
n = 604, pseudo-R² = 0.329, AIC = 595.60
Clustered SEs by institution. GPA mean-centered.
────────────────────────────────────────────────────────────


,OR,CI_95,p_value,sig
GPA (mean-centered),3.180,1.053 – 9.607,0.040,*
Female,1.228,0.756 – 1.995,0.407,
Pell Grant,1.312,0.966 – 1.781,0.082,
Veteran,2.173,0.819 – 5.766,0.119,
Student Year,1.168,0.925 – 1.474,0.191,
Graduate Student,0.513,0.202 – 1.302,0.160,
Military Affiliation,0.186,0.045 – 0.771,0.020,*



LPM vs. Logit comparison
───────────────────────────────────────────────────────


,Coef (LPM),sig (LPM),OR (Logit),sig (Logit)
GPA (mean-centered),0.149,,3.180,*
Female,0.025,,1.228,
Pell Grant,0.031,,1.312,
Veteran,0.092,,2.173,
Student Year,0.020,,1.168,
Graduate Student,-0.077,,0.513,
Military Affiliation,-0.220,*,0.186,*



Predictors where significance differs:


,Coef (LPM),sig (LPM),OR (Logit),sig (Logit)
GPA (mean-centered),0.149,,3.180,*


### 2.3 Sensitivity — Exclusion Threshold Robustness

Refit the LPM restricting to institutions with acceptance rates between 10% and 90%.

In [14]:
strict_insts = inst_stats[
    (inst_stats['acceptance_rate'] > 0.10) &
    (inst_stats['acceptance_rate'] < 0.90)
]['Institution']

df_strict = df_eligible[df_eligible['Institution'].isin(strict_insts)].copy()
# GPA already centered on the full eligible sample mean

lpm_strict, _ = fit_fe_model(df_strict, OUTCOME, COVARS, model='ols')
tbl_strict    = lpm_table(lpm_strict, COVARS)

sens = pd.DataFrame({
    'Coef (0/100)': lpm_tbl['Coef'],
    'sig (0/100)':  lpm_tbl['sig'],
    'Coef (10/90)': tbl_strict['Coef'],
    'sig (10/90)':  tbl_strict['sig'],
})

print(f'Threshold 0/100  : n={lpm_result.nobs:.0f}, {df_competitive["Institution"].nunique()} institutions')
print(f'Threshold 10/90  : n={lpm_strict.nobs:.0f}, {df_strict["Institution"].nunique()} institutions')
print('─' * 55)
display(sens.round(3))

changed = sens[sens['sig (0/100)'] != sens['sig (10/90)']]
if len(changed):
    print('\nPredictors where significance changes across thresholds:')
    display(changed.round(3))
else:
    print('\nNo predictors change significance across thresholds.')

Threshold 0/100  : n=604, 28 institutions
Threshold 10/90  : n=466, 23 institutions
───────────────────────────────────────────────────────


,Coef (0/100),sig (0/100),Coef (10/90),sig (10/90)
GPA (mean-centered),0.149,,0.182,
Female,0.025,,0.038,
Pell Grant,0.031,,0.046,
Veteran,0.092,,0.119,
Student Year,0.020,,0.026,
Graduate Student,-0.077,,-0.064,
Military Affiliation,-0.220,*,-0.297,*



No predictors change significance across thresholds.
